# Damped harmonic motion and mechanical energy

Solve a linear mass-spring oscillator released from rest. Compare three
damping rates in one broadcasted calculation, then check energy loss.
Run the setup in [examples/README.md](README.md) first.

The model is
$$
\ddot x+2\gamma\dot x+\omega_0^2x=0,\qquad
\omega_0=\sqrt{k/m},\qquad\omega_d=\sqrt{\omega_0^2-\gamma^2}.
$$
Here $m$ is mass, $k$ is spring stiffness, and $\gamma$ is a damping rate.
The familiar drag coefficient is $b=2m\gamma$. We use linear damping and
$0\leq\gamma<\omega_0$; critical and overdamped motion need other formulas.

Reference: [OpenStax, Damped Oscillations](https://openstax.org/books/university-physics-volume-1/pages/15-5-damped-oscillations).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from unitx.si import si

m = 0.2 * si.kg  # mass
k = 20.0 * si.N / si.m  # spring stiffness
x_0 = 2.0 * si.cm  # initial displacement
gamma = np.asarray([0.0, 0.25, 1.0]) / si.s  # damping rate
t = np.linspace(0, 8, 1601)[:, None] * si.s  # time
omega_0 = (k / m) ** 0.5
assert np.all(gamma < omega_0)
omega_d = np.sqrt(omega_0**2 - gamma**2)
print('Natural angular frequency:', omega_0)

## Apply the initial conditions explicitly

With $x(0)=x_0$ and $\dot x(0)=0$,
$$
x=x_0e^{-\gamma t}\left[\cos(\omega_dt)
  +\frac{\gamma}{\omega_d}\sin(\omega_dt)\right],\qquad
v=-\frac{x_0\omega_0^2}{\omega_d}e^{-\gamma t}\sin(\omega_dt).
$$
The sine term is needed for release from rest at nonzero damping.
Both trigonometric and exponential arguments are dimensionless quantities.

In [ ]:
phase = omega_d * t
decay = np.exp(-gamma * t)
x = x_0 * decay * (np.cos(phase) + gamma / omega_d * np.sin(phase))
v = -x_0 * omega_0**2 / omega_d * decay * np.sin(phase)
E = (m * v**2 / 2 + k * x**2 / 2).to(si.mJ)  # mechanical energy
E_0 = (k * x_0**2 / 2).to(si.mJ)

## Inspect motion and energy

The energy is $E=mv^2/2+kx^2/2$ and obeys
$dE/dt=-2m\gamma v^2\leq0$. Its instantaneous decay is not generally a
pure exponential, so the plot uses the calculated energy itself.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), layout='constrained')
labels = [f'gamma = {rate}' for rate in gamma]
axes[0].plot(t.value[:, 0], x.value)
axes[0].set(xlabel=f'Time [{t.unit}]', ylabel=f'Displacement [{x.unit}]', title='Release from rest')
axes[0].legend(labels)
axes[1].plot(t.value[:, 0], E.value)
axes[1].set(
    xlabel=f'Time [{t.unit}]', ylabel=f'Mechanical energy [{E.unit}]', title='Energy dissipation'
)
axes[1].legend(labels)
plt.show()

## Check the initial state and conservative limit

The undamped column must conserve energy. The other columns must not gain
energy; the small numerical allowance below only covers rounding.

In [ ]:
assert x.shape == (1601, 3)
assert np.all(x[0].isclose(x_0))
assert np.all(v[0].isclose(0 * si.m / si.s, abs_tol=1e-12 * si.m / si.s))
assert np.all(E[:, 0].isclose(E_0, rel_tol=1e-12))
assert np.all(np.diff(E.value, axis=0) <= 1e-12)
print('Initial energy:', E_0)
print('Final energies:', E[-1])